# 🔍 Attention Visualization

## Seeing Attention Instead of Just Reading About It

In this notebook, we will visualize how **Self-Attention** works.

Instead of looking at complicated sentences or large models, we will use a
tiny example so that we can actually see what the model is doing.

Our journey:

**Sentence**
↓
**Token Embeddings**
↓
**Queries, Keys, Values**
↓
**Attention Scores**
↓
**Softmax**
↓
**Attention Weights**
↓
**Attention Heatmap**

The goal is not to train a large Transformer.

The goal is to understand what happens **inside one attention mechanism**.

In [ ]:
import math
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Make the results reproducible
torch.manual_seed(42)

print("PyTorch version:", torch.__version__)

## 1. Our Example Sentence

Let's use a simple sentence:

> "The cat sat on the mat"

We will represent each word as a token.

For simplicity, we will use randomly generated embeddings.

In a real Transformer, these vectors would be learned during training.

In [ ]:
sentence = "The cat sat on the mat"

tokens = sentence.split()

print("Tokens:")
print(tokens)

print("\nNumber of tokens:", len(tokens))

## 2. Creating Token Embeddings

A Transformer does not directly work with words.

Each token is converted into a vector.

For this visualization, we will use:

- 6 tokens
- 8-dimensional embeddings

So our input will have the shape:

```text
(number of tokens, embedding dimension)

(6, 8)

In [ ]:
embedding_dim = 8

# Create a random embedding for each token
embeddings = torch.randn(
    len(tokens),
    embedding_dim
)

print("Embedding shape:", embeddings.shape)
print("\nEmbeddings:")
print(embeddings)

## 3. Queries, Keys and Values

Now we reach one of the most important ideas in Attention.

Every token is transformed into three vectors:

### Query (Q)
"What am I looking for?"

### Key (K)
"What information do I contain?"

### Value (V)
"What information should I pass along?"

For a real Transformer, Q, K and V are produced using learned
linear transformations.

For this demonstration, we will create them using simple matrices.

In [ ]:
# Create random weight matrices
W_q = torch.randn(embedding_dim, embedding_dim)
W_k = torch.randn(embedding_dim, embedding_dim)
W_v = torch.randn(embedding_dim, embedding_dim)

# Create Queries, Keys and Values
Q = embeddings @ W_q
K = embeddings @ W_k
V = embeddings @ W_v

print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)

## 4. Calculating Attention Scores

Now every token asks:

> "How relevant is every other token to me?"

We calculate this using:

$$
QK^T
$$

This gives us a matrix where:

- Rows = the token asking the question
- Columns = the token being considered

For 6 tokens, we get:

```text
6 × 6

In [ ]:
# Calculate raw attention scores
scores = Q @ K.T

print("Attention score shape:", scores.shape)

print("\nAttention scores:")
print(scores)

## 5. Scaling the Scores

The original Transformer does not directly apply Softmax to `QKᵀ`.

It first divides the scores by:

$$
\sqrt{d_k}
$$

This is called **Scaled Dot-Product Attention**.

Why?

Because when the vectors become large, the dot products can also become
large. Very large values can make Softmax extremely sharp.

Scaling helps keep the values in a more useful range.

In [ ]:
d_k = Q.shape[-1]

scaled_scores = scores / math.sqrt(d_k)

print("Scaled attention scores:")
print(scaled_scores)

## 6. Softmax → Attention Weights

Now we apply Softmax to each row.

Softmax converts the scores into probabilities.

For each token:

> "How much attention should I give to every other token?"

Each row therefore adds up to approximately **1**.

For example:

```text
The → [0.10, 0.20, 0.05, 0.30, 0.15, 0.20]

In [ ]:
attention_weights = F.softmax(
    scaled_scores,
    dim=-1
)

print("Attention weights:")
print(attention_weights)

print("\nRow sums:")
print(attention_weights.sum(dim=-1))

## 7. Visualizing Attention

This is the important part.

The matrix below shows:

**Row → token doing the attending**

**Column → token receiving attention**

A brighter/darker cell represents a stronger attention weight.

This gives us a visual representation of:

> "Which words are looking at which other words?"

In [ ]:
plt.figure(figsize=(9, 7))

plt.imshow(
    attention_weights.detach().numpy(),
    aspect="auto"
)

plt.xticks(
    range(len(tokens)),
    tokens
)

plt.yticks(
    range(len(tokens)),
    tokens
)

plt.xlabel("Token being attended to")
plt.ylabel("Token doing the attending")

plt.title("Self-Attention Heatmap")

plt.colorbar(
    label="Attention Weight"
)

plt.tight_layout()
plt.show()

# 8. Understanding One Token

Let's choose one token and inspect where it is looking.

For example:

> "cat"

We can extract the attention distribution for `"cat"`.

This lets us ask:

> "When the model processes the word `cat`, which other words does it
> consider important?"

In [ ]:
token_to_inspect = "cat"

token_index = tokens.index(token_to_inspect)

cat_attention = attention_weights[token_index]

print(f"Attention from '{token_to_inspect}':\n")

for token, weight in zip(tokens, cat_attention):
    print(f"{token:>5} → {weight.item():.4f}")

In [ ]:
plt.figure(figsize=(8, 4))

plt.bar(
    tokens,
    cat_attention.detach().numpy()
)

plt.xlabel("Token")
plt.ylabel("Attention Weight")

plt.title(
    f"Where does '{token_to_inspect}' pay attention?"
)

plt.tight_layout()
plt.show()

# 9. Producing the Final Attention Output

Attention does not stop at calculating the weights.

We use the weights to create a weighted combination of the Values.

The equation is:

$$
Attention(Q,K,V)
=
softmax
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)V
$$

In code:

```text
Attention Weights × V

In [ ]:
attention_output = attention_weights @ V

print("Attention output shape:")
print(attention_output.shape)

print("\nAttention output:")
print(attention_output)

# 🎯 The Complete Journey

We have just implemented the core mathematical idea behind
Self-Attention.

```text
             Token Embeddings
                    │
                    ▼
              ┌───────────┐
              │ Q   K   V │
              └─────┬─────┘
                    │
                    ▼
                  QKᵀ
                    │
                    ▼
              Scale by √dₖ
                    │
                    ▼
                 Softmax
                    │
                    ▼
           Attention Weights
                    │
                    ▼
              × Values (V)
                    │
                    ▼
           Attention Output

# 🧠 What Did We Actually Learn?

### 1. Attention is not magic

It is a sequence of mathematical operations:

**Dot Product → Scaling → Softmax → Weighted Sum**

### 2. Every token can look at every other token

This is why Self-Attention can capture relationships between distant words.

### 3. Q, K and V have different jobs

| Vector | Intuition |
|---|---|
| Query | What am I looking for? |
| Key | What information do I contain? |
| Value | What information should I provide? |

### 4. The heatmap is a window into the model

Instead of only seeing the final output, we can visualize the attention
weights and inspect which tokens interact with each other.

### 5. This is only ONE attention head

The original Transformer uses **multiple attention heads**.
